In [1]:
# LotArea
# SaleCondition
# OverallQual
# BsmtQual
# YearBuilt
# ExterQual
# GrLivArea
# TotalBsmtSF
# GarageCars
# KitchenQual
# Neighborhood
# Condition1

# This set balances continuous variables (LotArea, GrLivArea, GarageArea, PoolArea, TotalBsmtSF) with categorical variables (Condition2, Utilities, Neighborhood, SaleType) and ordinal quality ratings (OverallQual, ExterQual).

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


df1 = pd.read_csv("train.csv")
df2 = pd.read_csv("test.csv")

df = pd.concat([df1, df2], axis=0)

In [2]:
# ------------------------
# Selected features
# ------------------------
features = [
    "LotArea",
    "SaleCondition",
    "OverallQual",
    "BsmtQual",
    "YearBuilt",
    "ExterQual",
    "GrLivArea",
    "TotalBsmtSF",
    "GarageCars",
    "KitchenQual",
    "Neighborhood",
    "Condition1"
]

X = df[features].copy()

# Apply log safely
y = np.log1p(df["SalePrice"].fillna(df["SalePrice"].median()))
from sklearn.impute import KNNImputer

# Example: assume df is your DataFrame
# Select the column(s) you want to impute
sale_price = df[["SalePrice"]]   # keep as DataFrame for KNNImputer

# Initialize KNN imputer (you can tune n_neighbors)
imputer = KNNImputer(n_neighbors=5)

# Fit and transform
sale_price_imputed = imputer.fit_transform(sale_price)

# Replace back into DataFrame
df["SalePrice"] = sale_price_imputed

# Apply log1p transform
y = np.log1p(df["SalePrice"])

# ------------------------
# Ordinal encoding (quality features)
# ------------------------
qual_map = {"Po":1, "Fa":2, "TA":3, "Gd":4, "Ex":5}

for col in ["BsmtQual", "ExterQual", "KitchenQual"]:
    X[col] = X[col].map(qual_map)

# ------------------------
# Fill missing values
# ------------------------
X["BsmtQual"] = X["BsmtQual"].fillna(0)

# ------------------------
# One-hot encode categoricals
# ------------------------
categorical_cols = ["SaleCondition", "Neighborhood", "Condition1"]
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# ------------------------
# Train-test split
# ------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (2335, 46)
Test shape: (584, 46)


In [3]:
df[features].isnull().sum()

LotArea           0
SaleCondition     0
OverallQual       0
BsmtQual         81
YearBuilt         0
ExterQual         0
GrLivArea         0
TotalBsmtSF       1
GarageCars        1
KitchenQual       1
Neighborhood      0
Condition1        0
dtype: int64

In [4]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

# ------------------------
# Model
# ------------------------
xgb_model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# ------------------------
# Train
# ------------------------
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

# ------------------------
# Predict & evaluate
# ------------------------
y_pred = xgb_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("XGBoost RMSE:", rmse)

ModuleNotFoundError: No module named 'xgboost'

In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# ------------------------
# Model
# ------------------------
rf_model = RandomForestRegressor(
    n_estimators=1000,
    max_depth=None,
    min_samples_split=3,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

# ------------------------
# Train
# ------------------------
rf_model.fit(X_train, y_train)

# ------------------------
# Predict & evaluate
# ------------------------
y_pred = rf_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Random Forest RMSE:", rmse)

Random Forest RMSE: 0.23914883077507554


In [52]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# --- Step 1: Select 12 features ---
features = [
    "LotArea", "SaleCondition", "OverallQual", "Utilities", "YearBuilt",
    "ExterQual", "GrLivArea", "TotalBsmtSF", "GarageCars", "PoolArea",
    "Neighborhood", "Condition1"
]

# --- Step 2: Subset dataset ---
X = df[features]
y_raw = df[["SalePrice"]]   # keep as DataFrame for imputation

# --- Step 3: KNN Imputation for target ---
imputer = KNNImputer(n_neighbors=5)
y_imputed = imputer.fit_transform(y_raw)
df["SalePrice"] = y_imputed

# --- Step 4: Log transform target ---
y = np.log1p(df["SalePrice"])

# --- Step 5: Encode categorical features ---
X_encoded = pd.get_dummies(X, drop_first=True)

# --- Step 6: Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

# --- Step 7: Train Gradient Forest (RandomForest) ---
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
rf_mse = math.sqrt(mean_squared_error(y_test, rf_preds))

# --- Step 8: Train XGBoost ---
xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_test)
import math
xgb_mse = math.sqrt(mean_squared_error(y_test, xgb_preds))

# --- Step 9: Print results ---
print("Random Forest MSE:", rf_mse)
print("XGBoost MSE:", xgb_mse)


Random Forest MSE: 0.24486172051732677
XGBoost MSE: 0.24504669651214311


In [53]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# --- Step 1: Select new features ---
features = [
    "MSSubClass", "HouseStyle", "RoofStyle", "Exterior1st", "MasVnrArea",
    "BsmtFinSF1", "HeatingQC", "CentralAir", "KitchenQual", "Fireplaces",
    "GarageType", "PavedDrive"
]

# --- Step 2: Subset dataset ---
X = df[features]
y_raw = df[["SalePrice"]]   # keep as DataFrame for imputation

# --- Step 3: KNN Imputation for target ---
imputer = KNNImputer(n_neighbors=5)
y_imputed = imputer.fit_transform(y_raw)
df["SalePrice"] = y_imputed

# --- Step 4: Log transform target ---
y = np.log1p(df["SalePrice"])

# --- Step 5: Encode categorical features ---
X_encoded = pd.get_dummies(X, drop_first=True)

# --- Step 6: Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

# --- Step 7: Train Gradient Forest (RandomForest) ---
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))

# --- Step 8: Train XGBoost ---
xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_test)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_preds))

# --- Step 9: Print results ---
print("Random Forest RMSE:", rf_rmse)
print("XGBoost RMSE:", xgb_rmse)


Random Forest RMSE: 0.263683989442117
XGBoost RMSE: 0.26243507389970444


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor

# Load your dataset (assuming it's named 'data.csv')
# df = pd.read_csv('train.csv')

# For demonstration, let's assume 'df' is already loaded
# Define target and selected features
target = 'SalePrice'
features = [
    'OverallQual', 'GrLivArea', 'TotalBsmtSF', 'GarageCars',
    '1stFlrSF', 'YearBuilt', 'FullBath', 'KitchenQual',
    'Neighborhood', 'YearRemodAdd', 'Fireplaces', 'LotArea'
]

features = [
    "LotArea",
    "SaleCondition",
    "OverallQual",
    "BsmtQual",
    "YearBuilt",
    "ExterQual",
    "GrLivArea",
    "TotalBsmtSF",
    "GarageCars",
    "KitchenQual",
    "Neighborhood",
    "Condition1"
]

X = df[features]
# Perform log transformation on y
y = np.log1p(df[target])
y = y.fillna(y.median())

# Identify numerical and categorical columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

# 1. Preprocessing Pipelines
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Random Forest Model
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# 3. XGBoost Model
xgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(n_estimators=100, learning_rate=0.05, random_state=42))
])

# Train and Predict
models = {'Random Forest': rf_model, 'XGBoost': xgb_model}
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    # Calculate RMSE on the LOG scale
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    results[name] = rmse

# Final Output
print("--- Final Model Performance (Log-RMSE) ---")
for model_name, score in results.items():
    print(f"{model_name}: {score:.4f}")

/tmp/ipykernel_5254/3234722839.py:46: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=['object']).columns.tolist()


--- Final Model Performance (Log-RMSE) ---
Random Forest: 0.2364
XGBoost: 0.2331


In [5]:
import joblib
joblib.dump(xgb_model, "pipeline.joblib")

['pipeline.joblib']

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor

# Assuming 'train.csv' is in your directory
# df = pd.read_csv('train.csv')

# 12 Alternative Features
features = [
    'MSZoning', 'Foundation', 'BsmtQual', 'BsmtExposure',
    'HeatingQC', 'CentralAir', 'GarageType', 'GarageFinish',
    'PavedDrive', 'OpenPorchSF', 'MasVnrArea', 'SaleType'
]

features = [
    "LotArea",
    "SaleCondition",
    "OverallQual",
    "BsmtQual",
    "YearBuilt",
    "ExterQual",
    "GrLivArea",
    "TotalBsmtSF",
    "GarageCars",
    "KitchenQual",
    "Neighborhood",
    "Condition1"
]

X = df[features]
# Log Transformation and Imputation on the Target
y = np.log1p(df['SalePrice'])

# Split numerical and categorical
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

# Preprocessing Pipelines
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Imputation
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Imputation
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Models
rf = Pipeline(steps=[
    ('prepro', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])

xgb = Pipeline(steps=[
    ('prepro', preprocessor),
    ('model', XGBRegressor(n_estimators=100, learning_rate=0.05, random_state=42))
])

# Execution
results = {}
for name, pipeline in [("Random Forest", rf), ("XGBoost", xgb)]:
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    results[name] = rmse

# Output
print("--- Alternative Features Performance (Log-RMSE) ---")
for model, score in results.items():
    print(f"{model}: {score:.4f}")

ModuleNotFoundError: No module named 'xgboost'

In [56]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor

# Load dataset
# df = pd.read_csv('train.csv')

# Define features and target
features = [
    'MSZoning', 'Foundation', 'BsmtQual', 'BsmtExposure',
    'HeatingQC', 'CentralAir', 'GarageType', 'GarageFinish',
    'PavedDrive', 'OpenPorchSF', 'MasVnrArea', 'SaleType'
]

X = df[features]
y = np.log1p(df['SalePrice']) # Log transformation on target

# --- THREE-WAY SPLIT (Train, Val, Test) ---
# First, split into Train+Val (80%) and Test (20%)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Second, split Train+Val into Train (75% of 80% = 60%) and Val (25% of 80% = 20%)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42
)

# Identify column types
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

# Preprocessing
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

# Define Models
models = {
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.05, random_state=42)
}

print(f"{'Model':<20} | {'Val RMSE':<10} | {'Test RMSE':<10}")
print("-" * 45)

for name, regressor in models.items():
    # Create full pipeline
    model_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', regressor)
    ])

    # 1. Fit on Train set
    model_pipeline.fit(X_train, y_train)

    # 2. Evaluate on Validation set (used for tuning/checking)
    val_preds = model_pipeline.predict(X_val)
    val_rmse = np.sqrt(mean_squared_error(y_val, val_preds))

    # 3. Final Evaluation on Test set (the "final exam")
    test_preds = model_pipeline.predict(X_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))

    print(f"{name:<20} | {val_rmse:.4f}     | {test_rmse:.4f}")

Model                | Val RMSE   | Test RMSE 
---------------------------------------------
Random Forest        | 0.2412     | 0.2698
XGBoost              | 0.2338     | 0.2608
